In [192]:
import pandas as pd
import warnings
import json
import os
warnings.filterwarnings('ignore')

In [193]:
# from google.colab import files
# uploaded = files.upload()

In [194]:
df = pd.read_csv('dataset.csv')

In [195]:
df.head()

,Mileage,Engine,Kerb Weight,Fuel,Transmission Type,Power,No. of Cylinders,Registration Year
0,18.9 kmpl,1197 cc,935 kg,Petrol,Manual,82 bhp,4.0,2015
1,19.81 kmpl,1086 cc,860 kg,Petrol,Manual,68.05 bhp,4.0,Apr 2015
2,15.6 kmpl,1196 cc,1090 kg,Petrol,Manual,70 bhp,4.0,Dec 2019
3,18.9 kmpl,1197 cc,1060 kg,Petrol,Manual,81.86 bhp,4.0,Jul 2017
4,25.44 kmpl,936 cc,1025 kg,Diesel,Manual,56.3 bhp,3.0,2015


In [196]:
df.shape

(3628, 8)

In [197]:
# Missing percentage count
for col in df.columns:
    missing_count = df[col].isnull().sum()
    missing_percentage = (missing_count / len(df)) * 100
    print(f"{col}: {missing_percentage:.2f}% missing")

Mileage: 13.67% missing
Engine: 0.88% missing
Kerb Weight: 8.32% missing
Fuel: 12.76% missing
Transmission Type: 0.08% missing
Power: 2.32% missing
No. of Cylinders: 0.61% missing
Registration Year: 0.14% missing


In [198]:
def handle_missing_values(df, col):
    missing_pct = df[col].isnull().mean() * 100
    print(f"{col}: {missing_pct:.2f}% missing", end=" → ")

    # Drop if more than 50% missing
    if missing_pct > 50:
        df.drop(columns=[col], inplace=True)
        print("❌ Dropped (>50% missing)")

    # Between 30% and 50% — drop rows
    elif missing_pct > 30:
        df.dropna(subset=[col], inplace=True)
        print("🗑️ Dropped rows (30-50% missing)")

    # Between 10% and 30% — fill with mean/mode + add indicator column
    elif missing_pct > 10:
        if df[col].dtype == 'object':
            fill_val = df[col].mode()[0]
            strategy = f"mode ({fill_val})"
        else:
            fill_val = df[col].mean()
            strategy = f"mean ({fill_val:.2f})"

        df[f'{col}_was_missing'] = df[col].isnull().astype(int)  # indicator column
        df[col].fillna(fill_val, inplace=True)
        print(f"⚠️ Filled with {strategy} + added indicator column (10-30% missing)")

    # Less than 10% — fill with median/mode
    elif missing_pct > 0:
        if df[col].dtype == 'object':
            fill_val = df[col].mode()[0]
            strategy = f"mode ({fill_val})"
        else:
            fill_val = df[col].median()
            strategy = f"median ({fill_val:.2f})"

        df[col].fillna(fill_val, inplace=True)
        print(f"✅ Filled with {strategy} (<10% missing)")

    else:
        print("✅ No missing values")

    return df


# Apply to all columns
cols_with_missing = df.columns[df.isnull().any()].tolist()

for col in cols_with_missing:
    df = handle_missing_values(df, col)

Mileage: 13.67% missing → ⚠️ Filled with mode (18.9 kmpl) + added indicator column (10-30% missing)
Engine: 0.88% missing → ✅ Filled with mode (1197 cc) (<10% missing)
Kerb Weight: 8.32% missing → ✅ Filled with mode (1100 kg) (<10% missing)
Fuel: 12.76% missing → ⚠️ Filled with mode (Petrol) + added indicator column (10-30% missing)
Transmission Type: 0.08% missing → ✅ Filled with mode (Manual) (<10% missing)
Power: 2.32% missing → ✅ Filled with mode (81.86 bhp) (<10% missing)
No. of Cylinders: 0.61% missing → ✅ Filled with median (4.00) (<10% missing)
Registration Year: 0.14% missing → ✅ Filled with mode (Jun 2022) (<10% missing)


In [199]:
df.isnull().sum()

,0
Mileage,0
Engine,0
Kerb Weight,0
Fuel,0
Transmission Type,0
Power,0
No. of Cylinders,0
Registration Year,0
Mileage_was_missing,0
Fuel_was_missing,0


In [200]:
df['Fuel'].value_counts()

,count
Fuel,
Petrol,2777
Diesel,783
CNG,68


1. Mileage

In [201]:
df['Mileage'] = df['Mileage'].str.extract(r'(\d+\.?\d*)').astype(float)

2. Engine

In [202]:
df['Engine'] = df['Engine'].str.extract(r'(\d+)').astype(float)

3. Weight

In [203]:
df['Kerb Weight'] = df['Kerb Weight'].str.extract(r'(\d+)').astype(float)
df['Kerb Weight'].fillna(df['Kerb Weight'].median(), inplace=True) # Fill NaNs reintroduced by str.extract

4. Power

In [204]:
df['Power'] = df['Power'].str.extract(r'(\d+\.?\d*)').astype(float)

5. Registration Year

In [205]:
df['Registration Year'] = pd.to_numeric(
    df['Registration Year'].str.extract(r'(\d{4})')[0],
    errors='coerce'
)

6. Transmission Type

In [206]:
df['Transmission Type'] = df['Transmission Type'].map({
                                'Automatic': 1,
                                'Manual': 0
                            })

7. Fuel

In [207]:
def encode_and_show_dropped(df, column, drop_first=True):
    dummies = pd.get_dummies(df[column], prefix=column, drop_first=drop_first)

    all_categories = df[column].unique()
    encoded_categories = [col.replace(f"{column}_", "") for col in dummies.columns]
    dropped_categories = [cat for cat in all_categories if cat not in encoded_categories]

    print(f"All categories:     {list(all_categories)}")
    print(f"Encoded categories: {encoded_categories}")
    print(f"Dropped categories: {dropped_categories}")

    return dummies

# usage
fuel_dummies = encode_and_show_dropped(df, 'Fuel', drop_first=True)
df = pd.concat([df.drop(columns=['Fuel']), fuel_dummies], axis=1)


All categories:     ['Petrol', 'Diesel', 'CNG']
Encoded categories: ['Diesel', 'Petrol']
Dropped categories: ['CNG']


In [208]:
df['Mileage'].fillna(df['Mileage'].median(), inplace=True)

In [209]:
df.columns

Index(['Mileage', 'Engine', 'Kerb Weight', 'Transmission Type', 'Power',
       'No. of Cylinders', 'Registration Year', 'Mileage_was_missing',
       'Fuel_was_missing', 'Fuel_Diesel', 'Fuel_Petrol'],
      dtype='object')

In [210]:
# Applying the standard scaling

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [211]:
cols_to_scale = ['Engine', 'Kerb Weight', 'Power', 'No. of Cylinders', 'Registration Year']
unscaled_features = ['Fuel_Diesel', 'Fuel_Petrol', 'Transmission Type', 'Mileage']

scaled_features_array = scaler.fit_transform(df[cols_to_scale])

scaled_features_df = pd.DataFrame(scaled_features_array, columns=cols_to_scale, index=df.index)

scaled_df = pd.concat([scaled_features_df, df[unscaled_features]], axis=1)
display(scaled_df.head())

,Engine,Kerb Weight,Power,No. of Cylinders,Registration Year,Fuel_Diesel,Fuel_Petrol,Transmission Type,Mileage
0,-0.548746,-0.808441,-0.606537,0.295687,-0.920567,False,True,0,18.90
1,-0.783231,-1.017053,-0.882866,0.295687,-0.920567,False,True,0,19.81
2,-0.550858,-0.377308,-0.844239,0.295687,0.198487,False,True,0,15.60
3,-0.548746,-0.460753,-0.609310,0.295687,-0.361040,False,True,0,18.90
4,-1.100103,-0.558106,-1.115616,-1.396350,-0.920567,True,False,0,25.44


# Training the Deep  Learnign Model

In [212]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense

In [213]:
X = scaled_df.drop(columns=['Mileage'])
y = df['Mileage']

In [214]:
model = Sequential()

model.add(Dense(10, activation='relu', input_dim=X.shape[1]))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='linear'))

In [215]:
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_15 (Dense)                │ (None, 10)             │            90 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 8)              │            88 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 187 (748.00 B)

 Trainable params: 187 (748.00 B)

 Non-trainable params: 0 (0.00 B)

In [216]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [217]:
X.isnull().sum()

,0
Engine,0
Kerb Weight,0
Power,0
No. of Cylinders,0
Registration Year,0
Fuel_Diesel,0
Fuel_Petrol,0
Transmission Type,0


In [219]:
model.compile(loss='mean_squared_error', optimizer='adam')
history = model.fit(X_train, y_train, epochs=100, batch_size=1, verbose=0, validation_data=(X_test, y_test))

In [220]:
from sklearn.metrics import r2_score

# Generate predictions on the test set
y_pred_test = model.predict(X_test)

# Calculate R2 Score
r2 = r2_score(y_test, y_pred_test)

print(f"R2 Score: {r2:.4f}")

# Display the first few predictions on the test set
print("\nFirst 5 predictions (test set):")
print(y_pred_test[:5])

# Display the first few actual values of y_test (Mileage)
print("\nFirst 5 actual y_test values (Mileage):")
print(y_test[:5])

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
R2 Score: 0.5862

First 5 predictions (test set):
[[28.4438  ]
 [17.904398]
 [20.227524]
 [12.583542]
 [18.500841]]

First 5 actual y_test values (Mileage):
602     26.49
1826    20.28
2404    18.90
3178    12.55
2507    20.36
Name: Mileage, dtype: float64
